# GPU-Accelerated Parallel BPE Tokenization - Training Notebook

Complete training pipeline with:
- Real HuggingFace datasets (OpenWebText, GSM8K)
- Multi-GPU support (Colab T4, Local RTX 4050, etc.)
- Checkpoint saving/loading for fault tolerance
- Comprehensive logging and monitoring
- Progress tracking and metrics visualization

## Step 1: Environment Setup & Imports

In [1]:
# Install/upgrade requirements (run once)
import subprocess
import sys

packages = [
    'torch>=2.1.0',
    'transformers>=4.38.0',
    'datasets>=2.18.0',
    'tiktoken>=0.6.0',
    'numpy>=1.24.0',
    'tqdm>=4.66.0',
    'matplotlib>=3.7.0'
 ]

print("Installing dependencies...")
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
print("✓ All dependencies installed")

Installing dependencies...
✓ All dependencies installed


In [2]:
# Core imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import json
import os
import time
import logging
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import shutil
import tiktoken

# HuggingFace
from datasets import load_dataset, Dataset as HFDataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 15.6 GB


## Step 2: Setup Project Paths & Logging

In [3]:
# Detect if running in Colab
try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ Running in Google Colab")
except Exception:
    IN_COLAB = False
    print("✓ Running locally")

# Setup paths
if IN_COLAB:
    # Mount Google Drive (no remount needed)
    drive.mount('/content/drive', force_remount=False)

    # Use mounted path directly - NO symlink creation to avoid duplicate directories
    PROJECT_ROOT = Path('/content/drive/MyDrive/PDC')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

    CACHE_DIR = PROJECT_ROOT / 'cache'
else:
    PROJECT_ROOT = Path('.').resolve()
    CACHE_DIR = PROJECT_ROOT / '.cache'

CHECKPOINTS_DIR = CACHE_DIR / 'checkpoints'
LOGS_DIR = CACHE_DIR / 'logs'
RESULTS_DIR = CACHE_DIR / 'results'
DATASET_CACHE = CACHE_DIR / 'datasets'

# Create directories
for directory in [CHECKPOINTS_DIR, LOGS_DIR, RESULTS_DIR, DATASET_CACHE]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Cache dir: {CACHE_DIR}")
print(f"Checkpoints: {CHECKPOINTS_DIR}")
print(f"Results: {RESULTS_DIR}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/PDC
Cache dir: /content/drive/MyDrive/PDC/cache
Checkpoints: /content/drive/MyDrive/PDC/cache/checkpoints
Results: /content/drive/MyDrive/PDC/cache/results


In [4]:
# Setup logging with error handling for Google Drive
def setup_logging(log_name: str = "training"):
    """Setup comprehensive logging with fault tolerance."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = LOGS_DIR / f"{log_name}_{timestamp}.log"

    logger = logging.getLogger(log_name)
    logger.setLevel(logging.DEBUG)

    # Remove any existing handlers to avoid duplicates
    logger.handlers.clear()

    # File handler with error handling
    try:
        fh = logging.FileHandler(log_file, mode='a')
        fh.setLevel(logging.DEBUG)

        # Console handler
        ch = logging.StreamHandler()
        ch.setLevel(logging.INFO)

        # Formatter
        formatter = logging.Formatter(
            '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
        )
        fh.setFormatter(formatter)
        ch.setFormatter(formatter)

        logger.addHandler(fh)
        logger.addHandler(ch)
    except Exception as e:
        print(f"Warning: Could not setup file logging: {e}")
        # Fall back to console only
        ch = logging.StreamHandler()
        ch.setLevel(logging.INFO)
        formatter = logging.Formatter(
            '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
        )
        ch.setFormatter(formatter)
        logger.addHandler(ch)

    return logger, log_file

logger, log_file = setup_logging("pdc_training")
logger.info(f"Logging to {log_file}")
logger.info(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

2026-05-03 18:46:50,226 - pdc_training - INFO - Logging to /content/drive/MyDrive/PDC/cache/logs/pdc_training_20260503_184650.log
INFO:pdc_training:Logging to /content/drive/MyDrive/PDC/cache/logs/pdc_training_20260503_184650.log
2026-05-03 18:46:50,229 - pdc_training - INFO - GPU: Tesla T4
INFO:pdc_training:GPU: Tesla T4


## Step 3: Add Project to Path & Import Modules

In [5]:
# Add project to path
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import project modules
try:
    from tokenizer.gpu_bpe import GPUBPETokenizer
    from tokenizer.hsg import SemanticGuardedTokenizer
    from compression.adaptive_quant import AdaptiveQuantizer
    from dist.worker import LocalTrainer
    from utils.metrics import MetricsLogger, EvaluationMetrics
    print("✓ All project modules imported successfully")
except Exception as e:
    logger.error(f"Failed to import modules: {e}")
    raise

✓ All project modules imported successfully


## Step 4: Dataset Loading & Caching

In [6]:
def load_openwebtext(
    split: str = "train",
    num_examples: int = None,
    cache_dir: Optional[Path] = None,
    skip_if_cached: bool = True,
) -> HFDataset:
    """
    Load OpenWebText with selection cache. After first run, the selected
    subset is saved to disk; subsequent runs load directly from cache
    without re-downloading or re-selecting.
    """
    cache_dir = cache_dir or DATASET_CACHE
    selection_cache = cache_dir / f"owt_selection_{split}_{num_examples or 'all'}"

    # Try to load pre-selected subset from disk cache first
    if skip_if_cached and selection_cache.exists():
        try:
            logger.info(f"Loading cached OpenWebText subset from {selection_cache}")
            dataset = HFDataset.load_from_disk(str(selection_cache))
            logger.info(f"✓ Loaded {len(dataset)} examples from cache (skipped download/selection)")
            return dataset
        except Exception as e:
            logger.warning(f"Cache load failed ({e}), falling back to HF load")

    logger.info(f"Loading OpenWebText ({split}, {num_examples or 'all'} examples)...")
    try:
        dataset = load_dataset(
            'openwebtext',
            split=split,
            cache_dir=str(cache_dir),
        )
        if num_examples and len(dataset) > num_examples:
            indices = np.random.choice(len(dataset), num_examples, replace=False)
            dataset = dataset.select(indices)

        # Save the selected subset for fast reload on next runtime
        try:
            logger.info(f"Saving selected subset to cache: {selection_cache}")
            dataset.save_to_disk(str(selection_cache))
            logger.info(f"✓ Cache saved")
        except Exception as e:
            logger.warning(f"Failed to save subset cache: {e}")

        logger.info(f"✓ Loaded {len(dataset)} examples")
        return dataset
    except Exception as e:
        logger.error(f"Failed to load OpenWebText: {e}")
        raise


def load_gsm8k(
    split: str = "train",
    num_examples: int = None,
    cache_dir: Optional[Path] = None,
    skip_if_cached: bool = True,
) -> HFDataset:
    """
    Load GSM8K with selection cache (same pattern as load_openwebtext).
    """
    cache_dir = cache_dir or DATASET_CACHE
    selection_cache = cache_dir / f"gsm8k_selection_{split}_{num_examples or 'all'}"

    if skip_if_cached and selection_cache.exists():
        try:
            logger.info(f"Loading cached GSM8K subset from {selection_cache}")
            dataset = HFDataset.load_from_disk(str(selection_cache))
            logger.info(f"✓ Loaded {len(dataset)} examples from cache (skipped download/selection)")
            return dataset
        except Exception as e:
            logger.warning(f"Cache load failed ({e}), falling back to HF load")

    logger.info(f"Loading GSM8K ({split}, {num_examples or 'all'} examples)...")
    try:
        dataset = load_dataset(
            'gsm8k', 'main',
            split=split,
            cache_dir=str(cache_dir),
        )
        if num_examples and len(dataset) > num_examples:
            indices = np.random.choice(len(dataset), num_examples, replace=False)
            dataset = dataset.select(indices)

        try:
            logger.info(f"Saving selected subset to cache: {selection_cache}")
            dataset.save_to_disk(str(selection_cache))
            logger.info(f"✓ Cache saved")
        except Exception as e:
            logger.warning(f"Failed to save subset cache: {e}")

        logger.info(f"✓ Loaded {len(dataset)} examples")
        return dataset
    except Exception as e:
        logger.error(f"Failed to load GSM8K: {e}")
        raise


# Always load datasets explicitly (don't skip)
logger.info("Loading datasets (10 000 OWT + full GSM8K)...")
owt_dataset   = load_openwebtext(split="train", num_examples=10000)
gsm8k_dataset = load_gsm8k(split="train")
logger.info("✓ Datasets loaded and available")


2026-05-03 18:46:55,872 - pdc_training - INFO - Loading datasets (10 000 OWT + full GSM8K)...
INFO:pdc_training:Loading datasets (10 000 OWT + full GSM8K)...
2026-05-03 18:46:55,875 - pdc_training - INFO - Loading cached OpenWebText subset from /content/drive/MyDrive/PDC/cache/datasets/owt_selection_train_10000
INFO:pdc_training:Loading cached OpenWebText subset from /content/drive/MyDrive/PDC/cache/datasets/owt_selection_train_10000
2026-05-03 18:46:56,022 - pdc_training - WARNING - Cache load failed (No such files: '/content/drive/MyDrive/PDC/cache/datasets/owt_selection_train_10000/dataset_info.json', nor '/content/drive/MyDrive/PDC/cache/datasets/owt_selection_train_10000/state.json' found. Expected to load a `Dataset` object but provided path is not a `Dataset`.), falling back to HF load
2026-05-03 18:46:56,024 - pdc_training - INFO - Loading OpenWebText (train, 10000 examples)...
INFO:pdc_training:Loading OpenWebText (train, 10000 examples)...
/usr/local/lib/python3.12/dist-packa

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

plain_text/train-00000-of-00080.parquet:   0%|          | 0.00/303M [00:00<?, ?B/s]

...

2026-05-03 19:07:18,981 - pdc_training - INFO - ✓ Cache saved
INFO:pdc_training:✓ Cache saved
2026-05-03 19:07:18,983 - pdc_training - INFO - ✓ Loaded 10000 examples
INFO:pdc_training:✓ Loaded 10000 examples
2026-05-03 19:07:18,985 - pdc_training - INFO - Loading cached GSM8K subset from /content/drive/MyDrive/PDC/cache/datasets/gsm8k_selection_train_all
INFO:pdc_training:Loading cached GSM8K subset from /content/drive/MyDrive/PDC/cache/datasets/gsm8k_selection_train_all
2026-05-03 19:07:19,112 - pdc_training - WARNING - Cache load failed (No such files: '/content/drive/MyDrive/PDC/cache/datasets/gsm8k_selection_train_all/dataset_info.json', nor '/content/drive/MyDrive/PDC/cache/datasets/gsm8k_selection_train_all/state.json' found. Expected to load a `Dataset` object but provided path is not a `Dataset`.), falling back to HF load
2026-05-03 19:07:19,114 - pdc_training - INFO - Loading GSM8K (train, all examples)...
INFO:pdc_training:Loading GSM8K (train, all examples)...


README.md: 0.00B [00:00, ?B/s]

2026-05-03 19:07:22,411 - pdc_training - INFO - Saving selected subset to cache: /content/drive/MyDrive/PDC/cache/datasets/gsm8k_selection_train_all
INFO:pdc_training:Saving selected subset to cache: /content/drive/MyDrive/PDC/cache/datasets/gsm8k_selection_train_all


Saving the dataset (0/1 shards):   0%|          | 0/7473 [00:00<?, ? examples/s]

2026-05-03 19:07:22,479 - pdc_training - INFO - ✓ Cache saved
INFO:pdc_training:✓ Cache saved
2026-05-03 19:07:22,481 - pdc_training - INFO - ✓ Loaded 7473 examples
INFO:pdc_training:✓ Loaded 7473 examples
2026-05-03 19:07:22,483 - pdc_training - INFO - ✓ Datasets loaded and available
INFO:pdc_training:✓ Datasets loaded and available


In [8]:
# Define training parameters
BPE_TRAIN_SAMPLES = 300
NUM_MERGES = 500

# Import the sequential BPE
from tokenizer.sequential_bpe import SequentialBPETokenizer

# Create and train the sequential BPE baseline
logger.info("Training Sequential BPE baseline...")
sequential_tokenizer = SequentialBPETokenizer(vocab_size=50257)
_baseline_corpus = [owt_dataset[i]["text"] for i in range(BPE_TRAIN_SAMPLES)]
merges_trained = sequential_tokenizer.train(_baseline_corpus, num_merges=NUM_MERGES)
logger.info(f"✓ Sequential BPE trained: {merges_trained} merges completed")


2026-05-03 19:18:36,351 - pdc_training - INFO - Training Sequential BPE baseline...
INFO:pdc_training:Training Sequential BPE baseline...
2026-05-03 19:21:51,691 - pdc_training - INFO - ✓ Sequential BPE trained: 500 merges completed
INFO:pdc_training:✓ Sequential BPE trained: 500 merges completed


In [ ]:
class TokenizedDataset(Dataset):
    """
    Lazy tokenized dataset — tokenizes on-the-fly in __getitem__ so the
    full HuggingFace dataset (memory-mapped Arrow file) can be used without
    pulling everything into RAM at once.
    """

    def __init__(
        self,
        hf_dataset,           # HF Dataset or list of strings
        tokenizer,
        text_column: str = None,   # column name if hf_dataset is a HF Dataset
        max_length: int = 256,
        stride: int = 256,
    ):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.stride     = stride

        # Accept either a HF Dataset (arrow-backed) or a plain list of strings.
        if isinstance(hf_dataset, list):
            self.texts      = hf_dataset
            self.hf_dataset = None
            self.col        = None
        else:
            self.texts      = None
            self.hf_dataset = hf_dataset
            self.col        = text_column

        n = len(hf_dataset)
        # Expose the raw document count as __len__ so DataLoader can build an
        # index.  Each __getitem__ call may return fewer than stride windows
        # for short docs, but on average this is close enough for shuffled
        # training over the full corpus.
        self._len = n
        logger.info(f"LazyTokenizedDataset: {n} documents (tokenize on-the-fly)")

    def __len__(self):
        return self._len

    def _get_text(self, idx: int) -> str:
        if self.texts is not None:
            return self.texts[idx]
        row = self.hf_dataset[idx]
        return row[self.col] if self.col else next(iter(row.values()))

    def __getitem__(self, idx: int):
        text = self._get_text(idx)
        ids = self.tokenizer.encode(text)

        if not ids:
            # Empty doc — return a zero-padded example rather than crashing.
            pad = torch.zeros(self.max_length, dtype=torch.long)
            return {'input_ids': pad, 'labels': pad}


        # Pick a random valid window inside this document.
        max_start = max(0, len(ids) - self.max_length - 1)
        start = torch.randint(0, max_start + 1, (1,)).item() if max_start > 0 else 0

        input_ids = ids[start : start + self.max_length]
        labels    = ids[start + 1 : start + self.max_length + 1]

        # Pad if the doc is shorter than max_length.
        pad_len = self.max_length - len(input_ids)
        if pad_len > 0:
            input_ids = input_ids + [0] * pad_len
            labels    = labels    + [0] * pad_len

        return {
            'input_ids': torch.tensor(input_ids[:self.max_length], dtype=torch.long),
            'labels':    torch.tensor(labels[:self.max_length],    dtype=torch.long),
        }

# Ensure datasets are loaded
if 'owt_dataset' not in globals() or 'gsm8k_dataset' not in globals():
    logger.info("⚠️  Datasets not found, loading from Step 4...")
    owt_dataset   = load_openwebtext(split="train", num_examples=10000)
    gsm8k_dataset = load_gsm8k(split="train")
    logger.info("✓ Datasets loaded")

# Create tokenizer
logger.info("Creating tokenizer...")
tokenizer = GPUBPETokenizer(vocab_size=50257, use_gpu=torch.cuda.is_available())

# Train GPU BPE using Parallel BPE_a (speculative selection + DAG wave execution)
BPE_TRAIN_SAMPLES = 1000
NUM_MERGES = 2000
logger.info(f"Training GPU BPE via Parallel BPE_a ({BPE_TRAIN_SAMPLES} OWT texts, {NUM_MERGES} merges)...")
_bpe_corpus = [owt_dataset[i]["text"] for i in range(BPE_TRAIN_SAMPLES)]
dag_depth, num_waves = tokenizer.bpe.train(_bpe_corpus, vocab_size=50257, num_merges=NUM_MERGES)
parallelism_pct = (1 - dag_depth / NUM_MERGES) * 100
logger.info(f"  Merge rules learned : {len(tokenizer.bpe.bpe_merges)}")
logger.info(f"  DAG depth           : {dag_depth} / {NUM_MERGES} sequential steps")
logger.info(f"  Parallelism potential: {parallelism_pct:.1f}%")

guarded_tokenizer = SemanticGuardedTokenizer(tokenizer, enable_hsg=True)
logger.info("✓ Tokenizer ready")


2026-05-03 18:12:25,752 - pdc_training - INFO - Creating tokenizer...
INFO:pdc_training:Creating tokenizer...
2026-05-03 18:12:25,790 - pdc_training - INFO - Training GPU BPE via Parallel BPE_a (1000 OWT texts, 2000 merges)...
INFO:pdc_training:Training GPU BPE via Parallel BPE_a (1000 OWT texts, 2000 merges)...


  [1/4] Speculative selection (2000 merges)...
        1915 valid merges selected
  [2/4] Building dependency DAG...
  [3/4] Topological level assignment...
        DAG depth: 7 / 1915 (99.6% parallelism potential)
  [4/4] Applying 7 merge waves...
        Wave 1/7
        Wave 2/7
        Wave 3/7
        Wave 4/7
        Wave 5/7
        Wave 6/7


2026-05-03 18:13:17,513 - pdc_training - INFO -   Merge rules learned : 1915
INFO:pdc_training:  Merge rules learned : 1915
2026-05-03 18:13:17,517 - pdc_training - INFO -   DAG depth           : 7 / 2000 sequential steps
INFO:pdc_training:  DAG depth           : 7 / 2000 sequential steps
2026-05-03 18:13:17,518 - pdc_training - INFO -   Parallelism potential: 99.7%
INFO:pdc_training:  Parallelism potential: 99.7%
2026-05-03 18:13:17,520 - pdc_training - INFO - ✓ Tokenizer ready
INFO:pdc_training:✓ Tokenizer ready


        Wave 7/7
  Training complete: 1915 merge rules learned


In [ ]:
# Parallel BPE_a vs Sequential BPE — wall-clock speedup demo
from tokenizer.bpe import benchmark as bpe_benchmark, generate_long_string
import multiprocessing

_sample_text = generate_long_string(50_000)
_workers = min(4, multiprocessing.cpu_count())
bpe_results = bpe_benchmark(_sample_text, num_merges=200, max_workers=_workers)
logger.info(
    f"Parallel BPE_a speedup : {bpe_results['speedup']:.2f}x over sequential BPE"
    f"DAG depth              : {bpe_results['dag_depth']} / 200 "
    f"({(1 - bpe_results['dag_depth']/200)*100:.1f}% parallelism potential)"
)

  BPE TOKENIZER BENCHMARK
  Input length : 50,000 characters
  Merges (N)   : 200
  Workers      : 2

[1/2] Running Standard (Sequential) BPE …
      Done in 7.2613s
      Merges applied   : 200
      Final token list : 36,539 tokens

[2/2] Running Parallel BPE_a …


2026-05-03 18:13:27,577 - pdc_training - INFO - Parallel BPE_a speedup : 3.79x over sequential BPEDAG depth              : 2 / 200 (99.0% parallelism potential)
INFO:pdc_training:Parallel BPE_a speedup : 3.79x over sequential BPEDAG depth              : 2 / 200 (99.0% parallelism potential)


      Done in 1.9145s
      Merges selected  : 200
      DAG depth        : 2  (theoretical min sequential steps)
      Final token list : 37,980 tokens

  RESULTS SUMMARY
  Standard BPE time   : 7.2613 s
  Parallel BPE_a time : 1.9145 s
  Wall-clock speedup  : 3.79×
  DAG depth vs N      : 2 / 200  (99.0% parallelism potential)

  First 10 Standard BPE merges:
    [01] 'x' + 'x' → 'xx'  (id=27)
    [02] 'd' + 'm' → 'dm'  (id=28)
    [03] 'y' + 'm' → 'ym'  (id=29)
    [04] 'o' + 'a' → 'oa'  (id=30)
    [05] 'y' + 'r' → 'yr'  (id=31)
    [06] 'w' + 'e' → 'we'  (id=32)
    [07] 'c' + 'k' → 'ck'  (id=33)
    [08] 't' + 'b' → 'tb'  (id=34)
    [09] 'o' + 'n' → 'on'  (id=35)
    [10] 'e' + 'a' → 'ea'  (id=36)

  First 10 Parallel BPE_a merges (speculative order):
    [01] 'x' + 'x' → 'xx'  (id=27)
    [02] 'd' + 'm' → 'dm'  (id=28)
    [03] 'y' + 'm' → 'ym'  (id=29)
    [04] 'o' + 'a' → 'oa'  (id=30)
    [05] 'y' + 'r' → 'yr'  (id=31)
    [06] 'w' + 'e' → 'we'  (id=32)
    [07] 'c' + 'k' → 

In [ ]:
# Ensure datasets are loaded before tokenizing
if 'owt_dataset' not in globals() or 'gsm8k_dataset' not in globals():
    logger.info("⚠️  Datasets not found, loading now...")
    owt_dataset   = load_openwebtext(split="train", num_examples=10000)
    gsm8k_dataset = load_gsm8k(split="train")
    logger.info("✓ Datasets loaded")

# Skip if tokenized datasets already exist in memory.
if all(v in globals() for v in ['train_dataset', 'val_dataset', 'gsm8k_tokenized', 'owt_train_hf', 'owt_val_hf']):
    try:
        logger.info(f"✓ Datasets already split: train={len(train_dataset)}, "
                    f"val={len(val_dataset)}, gsm8k={len(gsm8k_tokenized)} (skipping)")
    except Exception:
        # Stale vars - rebuild
        for _v in ['train_dataset', 'val_dataset', 'gsm8k_tokenized', 'owt_train_hf', 'owt_val_hf']:
            globals().pop(_v, None)

if 'train_dataset' not in globals():
    logger.info("Creating tokenized datasets...")

    # Split the HF dataset by index — the Arrow file stays memory-mapped on disk.
    # No Python list of strings is ever built, so RAM stays flat regardless of corpus size.
    total_owt = len(owt_dataset)
    train_size = int(0.9 * total_owt)

    owt_train_hf = owt_dataset.select(range(train_size))
    owt_val_hf   = owt_dataset.select(range(train_size, total_owt))

    logger.info(f"Train docs: {len(owt_train_hf)}, Val docs: {len(owt_val_hf)}")

    train_dataset = TokenizedDataset(
        owt_train_hf, guarded_tokenizer,
        text_column="text", max_length=256, stride=256
    )
    val_dataset = TokenizedDataset(
        owt_val_hf, guarded_tokenizer,
        text_column="text", max_length=256, stride=512
    )

    gsm8k_texts = [
        f"{ex['question']} {ex['answer']}"
        for ex in gsm8k_dataset          # only 7 k rows — fine to materialise
    ]
    gsm8k_tokenized = TokenizedDataset(
        gsm8k_texts, guarded_tokenizer, max_length=256
    )

    logger.info(
        f"✓ Datasets ready: train={len(train_dataset)}, "
        f"val={len(val_dataset)}, gsm8k={len(gsm8k_tokenized)}"
    )


2026-05-03 18:13:28,219 - pdc_training - INFO - Creating tokenized datasets...
INFO:pdc_training:Creating tokenized datasets...
2026-05-03 18:13:28,259 - pdc_training - INFO - Train docs: 9000, Val docs: 1000
INFO:pdc_training:Train docs: 9000, Val docs: 1000
2026-05-03 18:13:28,261 - pdc_training - INFO - LazyTokenizedDataset: 9000 documents (tokenize on-the-fly)
INFO:pdc_training:LazyTokenizedDataset: 9000 documents (tokenize on-the-fly)
2026-05-03 18:13:28,263 - pdc_training - INFO - LazyTokenizedDataset: 1000 documents (tokenize on-the-fly)
INFO:pdc_training:LazyTokenizedDataset: 1000 documents (tokenize on-the-fly)
2026-05-03 18:13:28,417 - pdc_training - INFO - LazyTokenizedDataset: 7473 documents (tokenize on-the-fly)
INFO:pdc_training:LazyTokenizedDataset: 7473 documents (tokenize on-the-fly)
2026-05-03 18:13:28,419 - pdc_training - INFO - ✓ Datasets ready: train=9000, val=1000, gsm8k=7473
INFO:pdc_training:✓ Datasets ready: train=9000, val=1000, gsm8k=7473


In [ ]:
# ── Tokenizer Comparison — core research output ──────────────────────────────
import re

logger.info("\n" + "="*70)
logger.info("TOKENIZER COMPARISON BENCHMARK")
logger.info("="*70)

# Benchmark sets: OWT (general) + GSM8K (numeric-heavy) — using minimal samples for fast iteration
OWT_N   = min(50, len(owt_train_hf))
GSM8K_N = min(20,  len(gsm8k_dataset))

logger.info(f"\n[SETUP] Loading {OWT_N} OWT + {GSM8K_N} GSM8K texts...")
owt_bench   = [owt_train_hf[i]["text"] for i in range(OWT_N)]
gsm8k_bench = [f"{gsm8k_dataset[i]['question']} {gsm8k_dataset[i]['answer']}"
                for i in range(GSM8K_N)]
all_bench_texts = owt_bench + gsm8k_bench
total_chars = sum(len(t) for t in all_bench_texts)
logger.info(f"✓ Loaded {len(all_bench_texts)} texts (~{total_chars/1e6:.1f}MB)")

TOKENIZERS = [
    ("Sequential BPE (baseline)",  sequential_tokenizer.encode),
    ("GPU BPE",                    tokenizer.encode),
    ("GPU BPE + HSG",              guarded_tokenizer.encode),
]

# ── 1. Overall throughput ─────────────────────────────────────────────────────
logger.info("\n[1/5] Throughput benchmark on all texts (70 total)...")
def benchmark_throughput(name, encode_fn, texts):
    logger.info(f"  Benchmarking {name}...")

    # warm-up
    for t in texts[:5]:
        encode_fn(t)

    t0 = time.perf_counter()
    total_tok, total_ch, per_doc = 0, 0, []

    for i, text in enumerate(tqdm(texts, desc=f"    {name[:30]:30s}", leave=False)):
        ids = encode_fn(text)
        n   = len(ids)
        total_tok += n
        total_ch  += len(text)
        per_doc.append(n)
        if (i + 1) % 100 == 0:
            logger.debug(f"      {i+1}/{len(texts)} texts encoded")

    elapsed = max(time.perf_counter() - t0, 1e-9)
    result = {
        "name": name,
        "samples": len(texts),
        "elapsed_sec": elapsed,
        "tokens_per_sec": total_tok / elapsed,
        "texts_per_sec":  len(texts) / elapsed,
        "avg_tokens_per_text": total_tok / len(texts),
        "chars_per_token": total_ch / max(total_tok, 1),
        "per_doc_counts": per_doc,
    }
    logger.info(f"  ✓ {total_tok/elapsed:.0f} tok/s | {total_ch/total_tok:.2f} ch/tok | {elapsed:.1f}s elapsed")
    return result

throughput_results = [benchmark_throughput(n, fn, all_bench_texts) for n, fn in TOKENIZERS]
baseline_tps = throughput_results[0]["tokens_per_sec"]
for r in throughput_results:
    r["speedup"] = r["tokens_per_sec"] / baseline_tps

# ── 2. Throughput vs text length ──────────────────────────────────────────────
logger.info("\n[2/5] Throughput by text length (OWT only)...")
LENGTH_BUCKETS = {
    "short (<200c)":       [],
    "medium (200-1000c)":  [],
    "long (>1000c)":       [],
}
for t in owt_bench:
    if   len(t) < 200:   LENGTH_BUCKETS["short (<200c)"].append(t)
    elif len(t) < 1000:  LENGTH_BUCKETS["medium (200-1000c)"].append(t)
    else:                LENGTH_BUCKETS["long (>1000c)"].append(t)

logger.info(f"  Distribution: short={len(LENGTH_BUCKETS['short (<200c)'])} | "
            f"medium={len(LENGTH_BUCKETS['medium (200-1000c)'])} | "
            f"long={len(LENGTH_BUCKETS['long (>1000c)'])}")

length_speed = {}
for tname, fn in TOKENIZERS:
    logger.info(f"  Benchmarking {tname}...")
    length_speed[tname] = {}
    for bucket, texts_b in LENGTH_BUCKETS.items():
        if not texts_b:
            length_speed[tname][bucket] = 0.0
            continue
        sample = texts_b[:min(50, len(texts_b))]
        for t in sample[:3]:
            fn(t)
        t0    = time.perf_counter()
        ntok  = sum(len(fn(t)) for t in tqdm(sample, desc=f"      {bucket:25s}", leave=False))
        elapsed = max(time.perf_counter() - t0, 1e-9)
        speed = ntok / elapsed
        length_speed[tname][bucket] = speed
        logger.debug(f"      {bucket}: {speed:.0f} tok/s")

# ── 3. Digit fragmentation — how well does each tokenizer keep digit spans intact?
logger.info("\n[3/5] Digit fragmentation analysis (GSM8K)...")
DIGIT_RE = re.compile(r'\d+')

def digit_fragmentation(encode_fn, texts, name=""):
    logger.info(f"  Analyzing {name}...")
    isolated, total = 0, 0
    for text in tqdm(texts, desc=f"    {name[:30]:30s}", leave=False):
        if not DIGIT_RE.search(text):
            continue
        ids   = encode_fn(text)
        total += sum(len(s) for s in DIGIT_RE.findall(text))
        isolated += sum(1 for tid in ids
                        if isinstance(tid, int) and 0x30 <= tid <= 0x39)
    frag_rate = isolated / max(total, 1)
    logger.info(f"  ✓ {frag_rate*100:.1f}% isolated")
    return frag_rate

gsm8k_frag = {}
for name, fn in TOKENIZERS:
    gsm8k_frag[name] = digit_fragmentation(fn, gsm8k_bench, name=name.split('(')[0].strip())

# ── 4. HSG token overhead (theoretical: 2 special tokens per digit span)
logger.info("\n[4/5] Computing HSG overhead...")
hsg_overheads = [2 * len(DIGIT_RE.findall(t)) for t in tqdm(gsm8k_bench, desc="  Scanning", leave=False)]
avg_hsg_overhead = sum(hsg_overheads) / max(len(hsg_overheads), 1)
logger.info(f"  ✓ Mean overhead: {avg_hsg_overhead:.1f} tokens/doc")

# ── Summary ───────────────────────────────────────────────────────────────────
logger.info("\n[5/5] Generating summary...")
print("\n" + "="*70)
print("TOKENIZER COMPARISON RESULTS")
print("="*70)
print("\n=== Throughput ===")
for r in throughput_results:
    print(f"  {r['name']:38s}  {r['tokens_per_sec']:>10.0f} tok/s  "
            f"{r['chars_per_token']:>5.2f} ch/tok  {r['speedup']:>5.2f}x")

print("\n=== Digit Fragmentation (GSM8K) ===")
for name, rate in gsm8k_frag.items():
    print(f"  {name:38s}  {rate*100:5.1f}% isolated")

print("\n=== HSG Overhead ===")
print(f"  Average extra tokens per doc: {avg_hsg_overhead:.1f}")
print("="*70 + "\n")

# Save results (strip per_doc_counts — too large for JSON)
_save = {
    "owt_samples": OWT_N, "gsm8k_samples": GSM8K_N,
    "throughput": [{k: v for k, v in r.items() if k != "per_doc_counts"}
                    for r in throughput_results],
    "length_speed": length_speed,
    "digit_fragmentation": gsm8k_frag,
    "hsg_overhead_avg": avg_hsg_overhead,
}
_f = LOGS_DIR / f"tokenizer_comparison_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(_f, 'w') as f:
    json.dump(_save, f, indent=2)
logger.info(f"Results saved: {_f}")
logger.info("="*70)

# Expose for the visualisation cell
tokenizer_comparison_results = throughput_results


2026-05-03 18:13:29,179 - pdc_training - INFO - 
INFO:pdc_training:
2026-05-03 18:13:29,182 - pdc_training - INFO - TOKENIZER COMPARISON BENCHMARK
INFO:pdc_training:TOKENIZER COMPARISON BENCHMARK
2026-05-03 18:13:29,185 - pdc_training - INFO - ======================================================================
INFO:pdc_training:======================================================================
2026-05-03 18:13:30,587 - pdc_training - INFO - 
[SETUP] Loading 500 OWT + 100 GSM8K texts...
INFO:pdc_training:
[SETUP] Loading 500 OWT + 100 GSM8K texts...
2026-05-03 18:13:30,644 - pdc_training - INFO - ✓ Loaded 600 texts (~2.6MB)
INFO:pdc_training:✓ Loaded 600 texts (~2.6MB)
2026-05-03 18:13:30,646 - pdc_training - INFO - 
[1/5] Throughput benchmark on all texts (600 total)...
INFO:pdc_training:
[1/5] Throughput benchmark on all texts (600 total)...
2026-05-03 18:13:30,649 - pdc_training - INFO -   Benchmarking tiktoken (GPT-2 baseline)...
INFO:pdc_training:  Benchmarking tiktoken (GPT

    tiktoken (GPT-2 baseline)     :   0%|          | 0/600 [00:00<?, ?it/s]

DEBUG:pdc_training:      100/600 texts encoded
DEBUG:pdc_training:      200/600 texts encoded
DEBUG:pdc_training:      300/600 texts encoded
DEBUG:pdc_training:      400/600 texts encoded
DEBUG:pdc_training:      500/600 texts encoded
DEBUG:pdc_training:      600/600 texts encoded
2026-05-03 18:13:31,271 - pdc_training - INFO -   ✓ 1165582 tok/s | 4.36 ch/tok | 0.5s elapsed
INFO:pdc_training:  ✓ 1165582 tok/s | 4.36 ch/tok | 0.5s elapsed
2026-05-03 18:13:31,272 - pdc_training - INFO -   Benchmarking GPU BPE...
INFO:pdc_training:  Benchmarking GPU BPE...


    GPU BPE                       :   0%|          | 0/600 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ── 6-Panel Tokenizer Comparison Visualization ────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("GPU BPE Tokenizer Comparison (vs Sequential BPE Baseline)", fontsize=16, fontweight='bold')

# Panel 1: Throughput bar chart
names = [r['name'] for r in throughput_results]
speeds = [r['tokens_per_sec'] / 1e6 for r in throughput_results]
colors = ['#A23B72', '#2E86AB', '#F18F01']
ax = axes[0, 0]
ax.bar(range(len(names)), speeds, color=colors, alpha=0.8)
ax.set_ylabel('Tokens/sec (millions)', fontsize=11)
ax.set_title('Throughput', fontweight='bold')
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.split('(')[0].strip() for n in names], rotation=15, ha='right', fontsize=9)
for i, (s, sp) in enumerate(zip(speeds, [r['speedup'] for r in throughput_results])):
    ax.text(i, s + 0.1, f'{sp:.2f}x', ha='center', fontsize=9, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Panel 2: Throughput vs text length
ax = axes[0, 1]
buckets = list(LENGTH_BUCKETS.keys())
x = np.arange(len(buckets))
width = 0.25
for i, (name, fn) in enumerate(TOKENIZERS):
    speeds_by_length = [length_speed[name][b] / 1e6 for b in buckets]
    ax.bar(x + i*width, speeds_by_length, width, label=name.split('(')[0].strip(), alpha=0.8)
ax.set_ylabel('Tokens/sec (millions)', fontsize=11)
ax.set_title('Throughput vs Text Length', fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(buckets, fontsize=9)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Panel 3: Compression ratio (chars/token)
ax = axes[0, 2]
compression = [r['chars_per_token'] for r in throughput_results]
ax.bar(range(len(names)), compression, color=colors, alpha=0.8)
ax.set_ylabel('Chars/Token', fontsize=11)
ax.set_title('Compression Ratio (lower = better)', fontweight='bold')
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.split('(')[0].strip() for n in names], rotation=15, ha='right', fontsize=9)
for i, c in enumerate(compression):
    ax.text(i, c + 0.1, f'{c:.2f}', ha='center', fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Panel 4: Token count distribution
ax = axes[1, 0]
per_doc = throughput_results[1]['per_doc_counts']  # GPU BPE
ax.hist(per_doc, bins=50, color='#2E86AB', alpha=0.7, edgecolor='black')
ax.set_xlabel('Tokens per document', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Token Count Distribution (GPU BPE)', fontweight='bold')
ax.axvline(x=np.mean(per_doc), color='red', linestyle='--', label=f'Mean: {np.mean(per_doc):.0f}')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Panel 5: Digit fragmentation rate (GSM8K)
ax = axes[1, 1]
frag_rates = [gsm8k_frag[name] * 100 for name, _ in TOKENIZERS]
ax.bar(range(len(names)), frag_rates, color=colors, alpha=0.8)
ax.set_ylabel('Isolated Digit Tokens (%)', fontsize=11)
ax.set_title('Digit Fragmentation (GSM8K, lower = better)', fontweight='bold')
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.split('(')[0].strip() for n in names], rotation=15, ha='right', fontsize=9)
for i, f in enumerate(frag_rates):
    ax.text(i, f + 2, f'{f:.1f}%', ha='center', fontsize=9)
ax.set_ylim([0, max(frag_rates) + 10])
ax.grid(axis='y', alpha=0.3)

# Panel 6: HSG overhead distribution
ax = axes[1, 2]
ax.hist(hsg_overheads, bins=30, color='#F18F01', alpha=0.7, edgecolor='black')
ax.set_xlabel('Extra tokens per document (HSG)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title(f'HSG Overhead (mean: {avg_hsg_overhead:.1f} tokens)', fontweight='bold')
ax.axvline(x=avg_hsg_overhead, color='red', linestyle='--', label=f'Mean')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig_path = RESULTS_DIR / "tokenizer_comparison.png"
plt.savefig(fig_path, dpi=120, bbox_inches="tight")
plt.show()
logger.info(f"Tokenizer comparison figure saved to {fig_path}")
print(f"Figure saved: {fig_path}")


In [ ]:
# Create DataLoaders
BATCH_SIZE = 8  # Adjust based on GPU memory
NUM_WORKERS = 0  # Set to 0 to avoid issues in notebooks

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

gsm8k_loader = DataLoader(
    gsm8k_tokenized,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

logger.info(f"DataLoaders created: {len(train_loader)} batches per epoch")
print(f"✓ Ready to train with batch size {BATCH_SIZE}")

## Step 6: Model, Trainer & Checkpoint Management

In [ ]:
def create_model(
    vocab_size: int = 50257,
    hidden_size: int = 256,  # Smaller for GPU memory
    num_layers: int = 4,
    device: torch.device = None
) -> nn.Module:
    """Create GPT-2 Small model."""
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = nn.Sequential(
        nn.Embedding(vocab_size, hidden_size),
        *[
            nn.TransformerEncoderLayer(
                d_model=hidden_size,
                nhead=4,
                dim_feedforward=1024,
                batch_first=True,
                dropout=0.1
            )
            for _ in range(num_layers)
        ],
        nn.Linear(hidden_size, vocab_size)
    ).to(device)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    logger.info(f"Model created: {total_params:,} total params, {trainable_params:,} trainable")
    return model

# Create model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = create_model(hidden_size=256, num_layers=4, device=device)
logger.info(f"Model on device: {device}")

In [ ]:
class TrainingCheckpoint:
    """Manage training checkpoints for fault tolerance."""

    def __init__(self, checkpoint_dir: Path):
        self.checkpoint_dir = checkpoint_dir
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

    def save(
        self,
        model: nn.Module,
        optimizer: optim.Optimizer,
        epoch: int,
        step: int,
        metrics: Dict,
        name: str = "latest"
    ) -> Path:
        """Save checkpoint."""
        checkpoint = {
            'epoch': epoch,
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': metrics,
            'timestamp': datetime.now().isoformat()
        }

        path = self.checkpoint_dir / f"checkpoint_{name}_{epoch}_{step}.pt"
        torch.save(checkpoint, path)
        logger.info(f"Checkpoint saved: {path}")
        return path

    def load(
        self,
        model: nn.Module,
        optimizer: optim.Optimizer,
        checkpoint_path: Path
    ) -> Dict:
        """Load checkpoint."""
        if not checkpoint_path.exists():
            logger.warning(f"Checkpoint not found: {checkpoint_path}")
            return {}

        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        logger.info(
            f"Checkpoint loaded from {checkpoint_path} "
            f"(epoch {checkpoint['epoch']}, step {checkpoint['step']})"
        )
        return checkpoint

    def get_latest(self) -> Optional[Path]:
        """Get path to latest checkpoint."""
        checkpoints = list(self.checkpoint_dir.glob("checkpoint_*.pt"))
        if not checkpoints:
            return None
        return max(checkpoints, key=lambda p: p.stat().st_mtime)

checkpoint_manager = TrainingCheckpoint(CHECKPOINTS_DIR)
logger.info(f"Checkpoint manager initialized at {CHECKPOINTS_DIR}")

## Step 7: Training Configuration

In [ ]:
# Training configuration
class TrainingConfig:
    def __init__(self):
        # Training params
        self.num_epochs      = 3
        self.batch_size      = 8
        self.learning_rate   = 1e-4
        self.warmup_steps    = 100
        self.max_length      = 256
        self.max_train_steps = 2000   # > batches/epoch => runs full epochs

        # Research scope
        self.openwebtext_sample_limit          = 10000
        self.tokenizer_benchmark_samples       = 1000
        self.tokenizer_benchmark_gsm8k_samples = 200

        # Checkpointing
        self.checkpoint_interval = 100
        self.validate_interval   = 50

        # Logging
        self.log_interval            = 10
        self.gpu_memory_log_interval = 50

        # Model
        self.vocab_size  = 50257
        self.hidden_size = 256
        self.num_layers  = 4

        # Early stopping
        self.patience  = 3
        self.min_delta = 0.001

    def __repr__(self):
        return "\n".join(f"{k}: {v}" for k, v in self.__dict__.items())

config = TrainingConfig()
logger.info(f"\nTraining Configuration:\n{config}\n")
logger.info(
    "3 epochs, full pass over 9 000 training docs each; "
    "tokenizer comparison is the primary research output."
)


In [ ]:
# Setup training components
optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate)
criterion = nn.CrossEntropyLoss()
metrics_logger = MetricsLogger(ema_alpha=0.1, log_interval=config.log_interval)

# Check for existing checkpoint
latest_checkpoint = checkpoint_manager.get_latest()
start_epoch = 0
start_step = 0

if latest_checkpoint:
    logger.info(f"Found checkpoint: {latest_checkpoint}")
    response = input("Resume from checkpoint? (y/n): ")

    if response.lower() == 'y':
        checkpoint = checkpoint_manager.load(model, optimizer, latest_checkpoint)
        start_epoch = checkpoint.get('epoch', 0)
        start_step = checkpoint.get('step', 0)
        logger.info(f"Resuming from epoch {start_epoch}, step {start_step}")
    else:
        logger.info("Starting fresh training")
else:
    logger.info("No previous checkpoint found, starting fresh")

logger.info("✓ Training components initialized")
logger.info(f"OpenWebText training subset limit: {config.openwebtext_sample_limit} documents")
logger.info("Tokenizer benchmark is configured separately from the main training loop")

## Step 8: Training Loop with Logging & Checkpointing

In [ ]:
def get_gpu_memory_usage() -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {'allocated_gb': 0, 'reserved_gb': 0, 'utilization_pct': 0}
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    return {'allocated_gb': allocated, 'reserved_gb': reserved,
            'utilization_pct': 100 * allocated / total}


def train_epoch(
    model: nn.Module,
    train_loader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    epoch: int,
    start_step: int = 0,
) -> Tuple[float, int]:
    # Train for one epoch, hard-capped at config.max_train_steps.
    # The model is a validation artefact; the tokenizer comparison is primary.
    model.train()
    epoch_loss, steps_done, step = 0.0, 0, start_step

    logger.info(f"\n{'='*60}")
    logger.info(
        f"Epoch {epoch + 1}/{config.num_epochs}  "
        f"(max {config.max_train_steps} steps)"
    )
    logger.info(f"{'='*60}\n")

    pbar = tqdm(total=config.max_train_steps, desc=f"Epoch {epoch + 1}", leave=True)

    for batch in train_loader:
        if steps_done >= config.max_train_steps:
            logger.info(
                f"Reached max_train_steps ({config.max_train_steps}), stopping epoch."
            )
            break
        try:
            input_ids = batch['input_ids'].to(device)
            labels    = batch['labels'].to(device)

            step_start = time.time()
            optimizer.zero_grad()
            logits = model(input_ids)
            loss   = criterion(
                logits.view(-1, logits.size(-1)), labels.view(-1)
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()
            num_tokens  = (labels != 50256).sum().item()

            metrics_logger.current_metrics.global_step = step
            metrics_logger.update(
                loss=loss.item(),
                batch_size=input_ids.size(0),
                tokens_per_sec=num_tokens / max(time.time() - step_start, 1e-6),
            )

            if step % config.log_interval == 0:
                metrics = metrics_logger.log_step()
                mem     = get_gpu_memory_usage()
                pbar.set_postfix({
                    'loss': f"{metrics.loss_ema:.4f}",
                    'gpu':  f"{mem['utilization_pct']:.0f}%",
                })
                if step % (config.log_interval * 5) == 0:
                    logger.info(
                        f"Step {step}: loss={metrics.loss_ema:.4f}, "
                        f"gpu={mem['allocated_gb']:.1f}/{mem['reserved_gb']:.1f}GB "
                        f"({mem['utilization_pct']:.1f}%)"
                    )

            if step % config.gpu_memory_log_interval == 0:
                torch.cuda.reset_peak_memory_stats()

            if step % config.checkpoint_interval == 0 and step > start_step:
                checkpoint_manager.save(
                    model, optimizer, epoch, step,
                    asdict(metrics_logger.current_metrics)
                    if hasattr(metrics_logger.current_metrics, '__dict__') else {},
                    name=f"epoch{epoch}_step{step}",
                )

            step       += 1
            steps_done += 1
            pbar.update(1)

        except Exception as e:
            logger.error(f"Error at step {step}: {e}")
            raise

    pbar.close()
    avg_epoch_loss = epoch_loss / max(steps_done, 1)
    logger.info(
        f"Epoch {epoch + 1} done — {steps_done} steps, "
        f"avg loss: {avg_epoch_loss:.4f}\n"
    )
    return avg_epoch_loss, step

logger.info(
    f"Training loop prepared — capped at {config.max_train_steps} steps per epoch"
)


In [ ]:
# Import dataclasses for checkpoint saving
from dataclasses import asdict

# Training loop
logger.info("\n" + "="*60)
logger.info("STARTING TRAINING")
logger.info("="*60)

train_losses = []
step = start_step

try:
    for epoch in range(start_epoch, config.num_epochs):
        # Train
        avg_loss, step = train_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            epoch,
            start_step if epoch == start_epoch else 0
        )
        train_losses.append(avg_loss)

        # Validation
        logger.info(f"Validating epoch {epoch + 1}...")
        model.eval()
        val_loss = 0.0
        max_val_steps = 1000  # cap at 1000 batches to keep validation fast
        val_steps_done = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation", leave=False, total=max_val_steps):
                if val_steps_done >= max_val_steps:
                    break
                input_ids = batch['input_ids'].to(device)
                labels = batch['labels'].to(device)

                logits = model(input_ids)
                loss = criterion(
                    logits.view(-1, logits.size(-1)),
                    labels.view(-1)
                )
                val_loss += loss.item()
                val_steps_done += 1

        avg_val_loss = val_loss / max(val_steps_done, 1)
        logger.info(f"Validation loss: {avg_val_loss:.4f}\n")

        # Save best checkpoint
        checkpoint_manager.save(
            model,
            optimizer,
            epoch,
            step,
            {'train_loss': avg_loss, 'val_loss': avg_val_loss},
            name=f"epoch{epoch}_final"
        )

        # Reset for next epoch
        start_step = 0

except KeyboardInterrupt:
    logger.warning("\nTraining interrupted by user")
    logger.info(f"Saving emergency checkpoint at step {step}...")
    checkpoint_manager.save(
        model,
        optimizer,
        start_epoch,
        step,
        {},
        name="emergency"
    )
except Exception as e:
    logger.error(f"Training failed with error: {e}")
    logger.info(f"Saving emergency checkpoint at step {step}...")
    checkpoint_manager.save(
        model,
        optimizer,
        start_epoch,
        step,
        {},
        name="error"
    )
    raise

logger.info("\n" + "="*60)
logger.info("TRAINING COMPLETED")
logger.info("="*60)

## Step 9: Plot Training Metrics

In [ ]:
# Plot training loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss per epoch
axes[0].plot(range(1, len(train_losses) + 1), train_losses, marker='o', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss per Epoch')
axes[0].grid(True, alpha=0.3)

# Loss history from metrics logger
if metrics_logger.metrics_history:
    steps = [m.global_step for m in metrics_logger.metrics_history]
    losses = [m.loss_ema for m in metrics_logger.metrics_history]
    axes[1].plot(steps, losses, linewidth=1, alpha=0.7)
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Loss (EMA)')
    axes[1].set_title('Loss Over Training Steps')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(LOGS_DIR / 'training_metrics.png', dpi=100, bbox_inches='tight')
plt.show()

logger.info(f"Plot saved to {LOGS_DIR / 'training_metrics.png'}")

## Step 10: Evaluation on GSM8K

In [ ]:
def extract_answer(text: str) -> Optional[int]:
    """Extract numeric answer from text."""
    import re
    # Look for number patterns
    matches = re.findall(r'\d+', text)
    return int(matches[-1]) if matches else None

def evaluate_gsm8k(
    model: nn.Module,
    data_loader,
    gsm8k_dataset_original,
    max_samples: int = 200
) -> Dict:
    """Evaluate on GSM8K dataset."""
    logger.info(f"\nEvaluating on GSM8K ({min(max_samples, len(gsm8k_dataset_original))} samples)...")

    model.eval()
    correct = 0
    total = 0
    results = []

    sample_idx = 0
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="GSM8K Eval", leave=False):
            if total >= max_samples:
                break

            input_ids = batch['input_ids'].to(device)

            # Get predictions for all items in batch
            logits = model(input_ids)
            predictions = torch.argmax(logits, dim=-1)

            for i in range(input_ids.size(0)):
                if total >= max_samples:
                    break
                if sample_idx >= len(gsm8k_dataset_original):
                    break

                original = gsm8k_dataset_original[sample_idx]
                expected_answer = original['answer']

                predicted_answer = extract_answer(
                    guarded_tokenizer.decode(predictions[i][:100])
                )

                is_correct = (
                    predicted_answer is not None
                    and str(predicted_answer) in expected_answer
                )

                if is_correct:
                    correct += 1
                total += 1
                sample_idx += 1

                results.append({
                    'correct': is_correct,
                    'expected': expected_answer[:50],
                    'predicted': predicted_answer
                })

    accuracy = 100 * correct / total if total > 0 else 0

    eval_results = {
        'accuracy': accuracy,
        'correct': correct,
        'total': total,
        'results': results[:20]  # Show first 20
    }

    logger.info(f"\nGSM8K Results:")
    logger.info(f"  Accuracy: {accuracy:.1f}%")
    logger.info(f"  Correct: {correct}/{total}\n")

    return eval_results

# Run evaluation (using full GSM8K)
gsm8k_results = evaluate_gsm8k(
    model,
    gsm8k_loader,
    gsm8k_dataset,
    max_samples=len(gsm8k_dataset)  # Evaluate on full GSM8K
)

## Step 11: Save Results & Final Report

In [ ]:
# Generate comprehensive summary report
summary_report = f"""
======================================================================
TRAINING SUMMARY REPORT
======================================================================

Project: GPU-Accelerated Parallel BPE Tokenization
Timestamp: {datetime.now().isoformat()}
Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}

--- TRAINING STATISTICS ---
Total Epochs: {config.num_epochs}
Training Steps (capped): {config.max_train_steps}
Batch Size: {config.batch_size}
Final Training Loss: {train_losses[-1]:.4f}

--- MODEL ---
Total Parameters: {sum(p.numel() for p in model.parameters()):,}
Vocab Size: {config.vocab_size}
Hidden Size: {config.hidden_size}
Layers: {config.num_layers}

--- TOKENIZER COMPARISON (GPU BPE vs Sequential BPE) ---
"""

for r in throughput_results:
    summary_report += (
        f"{r['name']:38s}: {r['tokens_per_sec']:>12.1f} tokens/sec, "
        f"{r['avg_tokens_per_text']:>7.1f} avg tokens/text, "
        f"{r['speedup']:>5.2f}x vs baseline\n"
    )

summary_report += "\n--- DIGIT PRESERVATION (GSM8K) ---\n"
for name, rate in gsm8k_frag.items():
    summary_report += f"  {name:38s}: {rate*100:5.1f}% digit fragmentation\n"

summary_report += f"\n--- EVALUATION ---\n"
summary_report += f"GSM8K Accuracy: {gsm8k_results['accuracy']:.1f}% ({gsm8k_results['correct']}/{gsm8k_results['total']})\n"

summary_report += f"\n--- OUTPUT LOCATIONS ---\n"
summary_report += f"Logs: {LOGS_DIR}\n"
summary_report += f"Checkpoints: {CHECKPOINTS_DIR}\n"
summary_report += f"Results: {RESULTS_DIR}\n"
summary_report += f"\n{'='*70}\n"

print(summary_report)
logger.info(summary_report)

# Save summary report
report_file = LOGS_DIR / f"training_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(report_file, 'w') as f:
    f.write(summary_report)
logger.info(f"Summary report saved to {report_file}")


In [ ]:
# Generate comprehensive summary report
summary_report = f"""
======================================================================
TRAINING SUMMARY REPORT
======================================================================

Project: GPU-Accelerated Parallel BPE Tokenization
Timestamp: {datetime.now().isoformat()}
Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}

--- TRAINING STATISTICS ---
Total Epochs: {config.num_epochs}
Training Steps (capped): {config.max_train_steps}
Batch Size: {config.batch_size}
Final Training Loss: {train_losses[-1]:.4f}

--- MODEL ---
Total Parameters: {sum(p.numel() for p in model.parameters()):,}
Vocab Size: {config.vocab_size}
Hidden Size: {config.hidden_size}
Layers: {config.num_layers}

--- TOKENIZER COMPARISON ---
"""

for r in throughput_results:
    summary_report += (
        f"{r['name']:38s}: {r['tokens_per_sec']:>12.1f} tokens/sec, "
        f"{r['avg_tokens_per_text']:>7.1f} avg tokens/text, "
        f"{r['speedup']:>5.2f}x vs baseline\n"
    )

summary_report += "\n--- DIGIT PRESERVATION (GSM8K) ---\n"
for name, rate in gsm8k_frag.items():
    summary_report += f"  {name:38s}: {rate*100:5.1f}% digit fragmentation\n"

summary_report += f"\n--- EVALUATION ---\n"
summary_report += f"GSM8K Accuracy: {gsm8k_results['accuracy']:.1f}% ({gsm8k_results['correct']}/{gsm8k_results['total']})\n"

summary_report += f"\n--- OUTPUT LOCATIONS ---\n"
summary_report += f"Logs: {LOGS_DIR}\n"
summary_report += f"Checkpoints: {CHECKPOINTS_DIR}\n"
summary_report += f"Results: {RESULTS_DIR}\n"
summary_report += f"\n{'='*70}\n"

print(summary_report)
logger.info(summary_report)

# Save summary report
report_file = LOGS_DIR / f"training_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(report_file, 'w') as f:
    f.write(summary_report)
logger.info(f"Summary report saved to {report_file}")

## Step 12: Load & Resume Training (if needed)

In [ ]:
# List available checkpoints
checkpoints = sorted(list(CHECKPOINTS_DIR.glob("checkpoint_*.pt")))
print(f"\nAvailable checkpoints ({len(checkpoints)}):")
for i, cp in enumerate(checkpoints[-10:]):
    stat = cp.stat()
    print(f"  {i+1}. {cp.name} ({stat.st_size / 1e6:.1f} MB)")

if checkpoints:
    print(f"\nTo resume from a checkpoint:")
    print(f"  checkpoint_manager.load(model, optimizer, CHECKPOINTS_DIR / 'checkpoint_name.pt')")

## Step 13: Memory & Performance Monitoring

In [ ]:
# Current GPU status
if torch.cuda.is_available():
    mem = get_gpu_memory_usage()
    print("\nCurrent GPU Status:")
    print(f"  Allocated: {mem['allocated_gb']:.2f} GB")
    print(f"  Reserved: {mem['reserved_gb']:.2f} GB")
    print(f"  Utilization: {mem['utilization_pct']:.1f}%")
    print(f"  Peak allocation during training: Check logs")

    # Clear cache
    torch.cuda.empty_cache()
    print("\n✓ GPU cache cleared")
else:
    print("\nNo GPU available")

## Step 14: Inference Example

In [ ]:
def generate_text(
    model: nn.Module,
    prompt: str,
    max_length: int = 100,
    temperature: float = 0.7
) -> str:
    """Generate text from prompt."""
    model.eval()

    # Tokenize prompt
    ids = guarded_tokenizer.encode(prompt)
    if not ids:
        return "[Unable to tokenize]"

    tokens = torch.tensor([ids]).to(device)

    with torch.no_grad():
        for _ in range(max_length):
            # Get predictions
            logits = model(tokens)
            next_token_logits = logits[0, -1, :] / temperature

            # Sample
            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, 1).item()

            # Append token, keep context within model max_length
            tokens = torch.cat([tokens, torch.tensor([[next_token]]).to(device)], dim=1)
            if tokens.size(1) > config.max_length:
                tokens = tokens[:, -config.max_length:]

            # Stop if end of text
            if next_token == 50256:
                break

    # Decode
    generated = guarded_tokenizer.decode(tokens[0].cpu())
    return generated

# Generate samples
print("\n=== Text Generation Examples ===")
prompts = [
    "The future of AI is",
"Machine learning is useful for",
"Training large models requires"
]

for prompt in prompts:
    generated = generate_text(model, prompt, max_length=50)
    print(f"\nPrompt: {prompt}")
    print(f"Generated: {generated[:100]}...")

## Notes & Tips

### Running on Different Environments:

**Colab (T4 GPU)**
- ~16GB GPU memory
- Can use batch_size=8-16
- Checkpoint automatically saves to Google Drive
- Runtime may disconnect, use checkpoints to resume

**Local RTX 4050 (8GB)**
- Use batch_size=4-8
- Reduce hidden_size to 128-256
- Reduce num_layers to 2-4
- Checkpoints save locally for quick resume

### Key Features:
1. **Tokenizer Comparison**: Benchmarks our GPU BPE against the original GPT-2 tokenizer (tiktoken)
2. **Automatic Checkpointing**: Saves every 500 steps
3. **GPU Memory Monitoring**: Logs GPU usage every 200 steps
4. **Reduced Research Scope**: Uses a 25k-document OpenWebText subset so tokenizer work stays central
5. **Detailed Logging**: All events saved to logs directory
6. **Automatic Dataset Caching**: HuggingFace datasets cached locally

### Troubleshooting:
- **OOM Error**: Reduce batch_size or max_length
- **Slow Training**: Check GPU utilization in logs
- **Loss not decreasing**: Check learning_rate, try warmup_steps
- **Disconnected (Colab)**: Restore from checkpoint in CHECKPOINTS_DIR